# Template 02: Data Conditioning

**Purpose:** Clean and condition assembled data

**Inputs:**
- data/01_assembled.parquet (from Template 01)

**Outputs:**
- data/02_conditioned.parquet

**Processing:**
- Set proper data types
- Handle missing values
- Basic feature validation

In [1]:
# Parameters (injected by papermill)
config_path = "config/car_coll/v1"

In [2]:
# Parameters
config_path = "config/car_liab/v1"


In [3]:
# Imports
import pandas as pd
import numpy as np
import yaml
import os
import sys
import gc
from pathlib import Path

# Setup environment
sys.path.insert(0, str(Path.cwd() / 'lib'))
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 02: DATA CONDITIONING")
print("########################################")

project_root = setup_notebook_environment()
print(f"\nProject root: {project_root}")

########################################
# STAGE 02: DATA CONDITIONING
########################################

Project root: /Users/Mach/dev/aps/code/26Dmodelv1


In [4]:
# Load config
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

# Get current machine ID and output path
pc_num = open('current.pc').read().strip()
pc_id = f'PC{pc_num}'  # current.pc has '3', config key is 'PC3'
output_base = cfg['machines'][pc_id]['paths']['output_path']
print(f"\n* Experiment: {cfg['experiment']['name']}")
print(f"* Target: {cfg['experiment']['target']}")


* Experiment: car_liab3p_v1
* Target: pp_bi


In [5]:
# Load checkpoint from Template 01
checkpoint_01 = f"{output_base}/data/01_assembled.parquet"
print(f"\n* Loading checkpoint: {checkpoint_01}")

data = pd.read_parquet(checkpoint_01)
print(f"  Shape: {data.shape}")
print(f"  Memory: {data.memory_usage(deep=True).sum() / 1e9:.2f} GB")


* Loading checkpoint: output/car_liab/v1/data/01_assembled.parquet


  Shape: (9089028, 115)


  Memory: 10.65 GB


In [6]:
# Check data types
print(f"\n* Data type distribution:")
print(data.dtypes.value_counts())


* Data type distribution:
int64      78
float64    24
object      6
str         6
uint32      1
Name: count, dtype: int64


In [7]:
# Check for missing values
missing_counts = data.isnull().sum()
missing_cols = missing_counts[missing_counts > 0]

if len(missing_cols) > 0:
    print(f"\n* Missing values found in {len(missing_cols)} columns:")
    for col, count in missing_cols.head(10).items():
        pct = (count / len(data)) * 100
        print(f"  {col}: {count:,} ({pct:.2f}%)")
    if len(missing_cols) > 10:
        print(f"  ... and {len(missing_cols) - 10} more")
else:
    print(f"\n* No missing values found")


* Missing values found in 15 columns:
  NumMajViol_raw: 7,461,920 (82.10%)
  NumMajinAcc_raw: 8,147,746 (89.64%)
  NumMinAcc_raw: 8,145,994 (89.62%)
  NumMinViol_raw: 7,461,920 (82.10%)
  NumSpdViol_raw: 8,146,489 (89.63%)
  late_payments_raw: 7,362,710 (81.01%)
  vc_seating_rows_raw: 1,793,454 (19.73%)
  BODY_STYLE_SEGMENT_BODY_TYPE: 7,045 (0.08%)
  Dep_factor: 7,045 (0.08%)
  veh_value_dep: 7,045 (0.08%)
  ... and 5 more


In [8]:
# Apply type conversions from config
type_conv_file = f"{config_path}/type_conversions.csv"

if os.path.exists(type_conv_file):
    print(f"\n* Applying type conversions from {type_conv_file}")
    conversions = pd.read_csv(type_conv_file)
    for _, row in conversions.iterrows():
        col = row['column_name']
        if col in data.columns and row['target_dtype'] == 'numeric':
            data[col] = pd.to_numeric(data[col], errors='coerce')
            print(f"  [OK] {col}: object -> numeric")
else:
    print(f"\n* No type conversions file found, skipping")

# Basic conditioning: Fill missing values with 0 for numeric features
# (More sophisticated strategies can be added later)
print(f"\n* Conditioning data...")

# Get numeric columns
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
print(f"  Numeric columns: {len(numeric_cols)}")

# Fill missing values in numeric columns
for col in numeric_cols:
    if data[col].isnull().any():
        data[col] = data[col].fillna(0)

print(f"  Missing values filled with 0")


* Applying type conversions from config/car_liab/v1/type_conversions.csv


  [OK] NumMajViol_raw: object -> numeric


  [OK] NumMajinAcc_raw: object -> numeric


  [OK] NumMinAcc_raw: object -> numeric


  [OK] NumMinViol_raw: object -> numeric


  [OK] NumSpdViol_raw: object -> numeric


  [OK] late_payments_raw: object -> numeric

* Conditioning data...
  Numeric columns: 109


  Missing values filled with 0


In [9]:
# Verify target variable exists
target = cfg['experiment']['target']
if target in data.columns:
    print(f"\n* Target variable '{target}' verified")
    print(f"  Non-null count: {data[target].notnull().sum():,}")
    print(f"  Mean: {data[target].mean():.4f}")
    print(f"  Median: {data[target].median():.4f}")
    print(f"  Min: {data[target].min():.4f}")
    print(f"  Max: {data[target].max():.4f}")
else:
    print(f"\n* ERROR: Target variable '{target}' not found!")
    print(f"  Available columns: {data.columns.tolist()[:20]}")


* Target variable 'pp_bi' verified
  Non-null count: 9,089,028
  Mean: 262.6851
  Median: 0.0000
  Min: 0.0000
  Max: 35587500.0000


In [10]:
# Validate output before saving
from utils import validate_critical_columns
validate_critical_columns(data, cfg, 'Stage 02 Output')

# Save conditioned checkpoint
checkpoint_02 = f"{output_base}/data/02_conditioned.parquet"
print(f"\n* Saving checkpoint: {checkpoint_02}")

data.to_parquet(checkpoint_02)
file_size = os.path.getsize(checkpoint_02) / 1e9

print(f"  Saved: {file_size:.2f} GB")
print(f"  Shape: {data.shape}")

[Stage 02 Output] [OK] Critical columns validated: vin_date, fold, pp_bi, ee_bi_imps

* Saving checkpoint: output/car_liab/v1/data/02_conditioned.parquet


  Saved: 0.83 GB
  Shape: (9089028, 115)


In [11]:
# Memory cleanup
print(f"\n* Cleaning memory...")
del data
gc.collect()
print(f"  Memory cleaned")


* Cleaning memory...
  Memory cleaned


In [12]:
print("\n########################################")
print("# STAGE 02: COMPLETE")
print("########################################")


########################################
# STAGE 02: COMPLETE
########################################
